# Exercise 1: Hydrogenation of a bimetallic nanoparticle

## Background

The hydrogen storage capacity of metallic nanoparticles depends on three key
structural parameters: **size**, **shape**, and **chemical composition**.
For AuPd bimetallic systems, even small changes in the Au:Pd ratio or in the
degree of chemical ordering can drastically change how much hydrogen the particle
absorbs at a given temperature and pressure.

In this exercise we use **grand-canonical Monte Carlo (GCMC)** to simulate
hydrogen adsorption on AuPd nanoparticles. The chemical potential $\mu$ of the
hydrogen reservoir controls the H loading, and by sweeping $\mu$ we can build
the adsorption isotherm $\langle N_H \rangle(\mu)$.

The interatomic interactions are described by a **GAP machine-learning
potential** trained on DFT data for the Au–Pd–H system, which gives
near-DFT accuracy at a fraction of the cost.

## Goals

1. Generate an AuPd nanoparticle using a realistic "cooking" recipe:
   anneal at high temperature → quench → relax → anneal.
2. Run GCMC to hydrogenate the cooked nanoparticle and monitor H uptake.
3. Compare your cooked structure against two reference structures:
   an ordered and a disordered truncated octahedron.
4. Analyse energy and H-concentration convergence.
5. Explore the effect of GCMC parameters on the result.
6. *(Optional)* Repeat for a larger nanoparticle ($N = 79$ atoms).

## mc.log column reference

The `mc.log` file written by TurboGAP has the following columns:

| Column | Quantity |
|--------|----------|
| 0 | MC step |
| 1 | mc_move |
| 2 | Accepted |
| 3 | E_trial (eV) |
| 4 | E_current (eV) |
| 5 | Total number of atoms |
| 6 | Species |
| 7 | Number of H atoms |


## Setup

Unpack the GAP potential files and import the required Python packages.
Run the cell below once at the start of the session.


In [1]:
import subprocess
subprocess.run(["tar", "-xzf", "gap_files.tar.xz"])

from ase.io import write, read
from weas_widget import WeasWidget
import numpy as np
import matplotlib.pyplot as plt


tar (child): gap_files.tar.xz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now


ModuleNotFoundError: No module named 'weas_widget'

---
## Part 1 — Nanoparticle preparation ("cooking")

A realistic nanoparticle is not a perfect crystal fragment.
We mimic experimental synthesis by running a four-step thermal protocol:

| Step | Script | Purpose |
|------|--------|---------|
| 1. Initial structure | `make_NP.py` | Place 25 Au/Pd atoms in a sphere |
| 2. Anneal at high temperature | `1.anneal.sh` | MD at 800 K — randomise atom positions and chemical order |
| 3. Quench | `2.quench.sh` | Rapid cooling to 100 K — freeze in a low-energy disordered state |
| 4. Relax | `3.relax.sh` | Energy minimisation — remove residual forces |
| 5. Anneal | `4.anneal2.sh` | MD at 600 K — allow local rearrangements towards equilibrium |

Each step reads the output of the previous one.
Visualise the structure after each step to see how it evolves.


### Step 1 — Initial structure

Run `python make_NP.py` in the terminal to generate the starting nanoparticle. The script places Au and Pd atoms on an FCC lattice in a 1:1 ratio, randomly distributed with roughly spherical shape.


In [ ]:
atoms = read("structures/atoms0.xyz")
print(f"Composition: {atoms.get_chemical_formula()}")
print(f"Number of atoms: {len(atoms)}")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


### Step 2 — Anneal at high temperature (800 K)

Copy and run the melt script:
```bash
cp scripts/1.anneal.sh .
bash 1.anneal.sh
```
The nanoparticle is heated to 800 K, below the melting temperature to favor crystallisation 


In [ ]:
atoms = read("structures/atoms1.xyz")
print(f"Energy after melt: {atoms.get_potential_energy():.3f} eV")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


### Step 3 — Quench (800 K → 100 K)

```bash
cp scripts/2.quench.sh .
bash 2.quench.sh
```
The particle is rapidly cooled. The disordered atomic arrangement from the melt
is preserved — this mimics a kinetically trapped, chemically disordered state.


In [ ]:
atoms = read("structures/atoms2.xyz")
print(f"Energy after quench: {atoms.get_potential_energy():.3f} eV")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


### Step 4 — Relax (energy minimisation)

```bash
cp scripts/3.relax.sh .
bash 3.relax.sh
```
A geometry optimisation removes residual forces from the MD.
The structure should have a lower energy than after the quench.

> **Question:** How does the energy per atom compare with your neighbours?
> A lower energy means a more stable chemical ordering was found during the melt.


In [ ]:
atoms = read("structures/atoms3.xyz")
E_relax = atoms.get_potential_energy()
N_atoms = len(atoms)
print(f"Energy after relax:          {E_relax:.3f} eV")
print(f"Energy per atom:             {E_relax/N_atoms:.4f} eV/atom")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


### Step 5 — Anneal (600 K)

```bash
cp scripts/4.anneal2.sh .
bash 4.anneal2.sh
```
A second, MD annealing run allows the particle to explore nearby
configurations and settle into a more stable structure.
This is the final "cooked" nanoparticle you will use for GCMC.


In [ ]:
atoms = read("structures/atoms4.xyz")
E_anneal = atoms.get_potential_energy()
print(f"Energy after anneal:         {E_anneal:.3f} eV")
print(f"Energy per atom:             {E_anneal/N_atoms:.4f} eV/atom")
print(f"Energy gain from anneal:     {E_relax - E_anneal:.3f} eV")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


---
## Part 2 — Grand-canonical Monte Carlo (GCMC)

We now run GCMC to insert hydrogen into the nanoparticle.
The simulation operates at fixed $\mu$, $V$, $T$ — the $\mu$VT ensemble.
At each MC step, TurboGAP randomly attempts one of:

- **Displacement** of a randomly chosen atom
- **Insertion** of a H atom at a random position
- **Removal** of a randomly chosen H atom

The acceptance probabilities follow the Metropolis criterion derived in the
lectures. The chemical potential $\mu$ controls the H reservoir — a higher
$\mu$ favours insertion and leads to a higher H concentration.

### GCMC-1 — your cooked nanoparticle


Copy and run the first GCMC script:
```bash
cp scripts/5.gcmc-1.sh .
bash 5.gcmc-1.sh
```
This runs GCMC on the annealed structure from Part 1.
Visualise the trajectory to see H atoms being inserted and removed.


In [ ]:
atoms = read("5.gcmc-1/mc_all.xyz", index=':')
print(f"Number of saved MC frames: {len(atoms)}")
print(f"H atoms in last frame:     "
      f"{atoms[-1].get_chemical_symbols().count('H')}")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


Plot the energy and H concentration as a function of MC step.
Check that both quantities have converged — if they are still drifting,
the simulation has not yet equilibrated.


In [ ]:
N = len(read("structures/atoms4.xyz"))-2  # number of metal atoms

data   = np.genfromtxt('5.gcmc-1/mc.log')
nsteps = data[:, 0]
ener   = data[:, 4]
cH     = data[:, 7]

fig, axs = plt.subplots(1, 2, figsize=(8, 4), layout='constrained')

axs[0].plot(nsteps, ener, color='steelblue', lw=0.8)
axs[0].set_xlabel('MC steps')
axs[0].set_ylabel('Energy (eV)')
axs[0].set_title('Total energy')

axs[1].plot(nsteps, cH / N * 100, color='darkorange', lw=0.8)
axs[1].set_xlabel('MC steps')
axs[1].set_ylabel('H / metal (%)')
axs[1].set_title('H concentration')

# Equilibrium estimate (last 50% of run)
half = len(cH) // 2
print(f"Mean H concentration (last 50%): {cH[half:].mean()/N*100:.1f} %")

plt.show()


### GCMC-2 — ordered truncated octahedron

We now compare against a reference structure: a **perfectly ordered** AuPd
truncated octahedron with 55 atoms (`PW55_AuPd.xyz`).
In this structure Au and Pd occupy well-defined sublattice sites.

```bash
cp scripts/6.gcmc-2.sh .
bash 6.gcmc-2.sh
```


In [ ]:
atoms = read("6.gcmc-2/mc_all.xyz", index=':')
print(f"Number of saved MC frames: {len(atoms)}")
print(f"H atoms in last frame:     "
      f"{atoms[-1].get_chemical_symbols().count('H')}")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


In [ ]:
N = 55

data   = np.genfromtxt('6.gcmc-2/mc.log')
nsteps = data[:, 0]
ener   = data[:, 4]
cH     = data[:, 7]

fig, axs = plt.subplots(1, 2, figsize=(8, 4), layout='constrained')

axs[0].plot(nsteps, ener, color='steelblue', lw=0.8)
axs[0].set_xlabel('MC steps')
axs[0].set_ylabel('Energy (eV)')
axs[0].set_title('Total energy — ordered NP')

axs[1].plot(nsteps, cH / N * 100, color='darkorange', lw=0.8)
axs[1].set_xlabel('MC steps')
axs[1].set_ylabel('H / metal (%)')
axs[1].set_title('H concentration — ordered NP')

half = len(cH) // 2
print(f"Mean H concentration (last 50%): {cH[half:].mean()/N*100:.1f} %")

plt.show()


### GCMC-3 — disordered truncated octahedron

Finally, compare against a **chemically disordered** truncated octahedron
(`PW79_dis_AuPd.xyz`) with the same size and shape but randomised
Au/Pd occupancy.

```bash
cp scripts/7.gcmc-3.sh .
bash 7.gcmc-3.sh
```

> **Question:** Does chemical ordering matter for hydrogen uptake?
> Compare the mean H concentrations from GCMC-2 and GCMC-3.
> What does this tell you about the role of local chemical environment?


In [ ]:
atoms = read("7.gcmc-3/mc_all.xyz", index=':')
print(f"Number of saved MC frames: {len(atoms)}")
print(f"H atoms in last frame:     "
      f"{atoms[-1].get_chemical_symbols().count('H')}")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


In [ ]:
N = 79

fig, axs = plt.subplots(1, 2, figsize=(8, 4), layout='constrained')
labels = ['ordered', 'disordered']
colors = ['steelblue', 'darkorange']

for idx, (folder, label, color) in enumerate(zip(
        ['6.gcmc-2', '7.gcmc-3'], labels, colors)):
    data   = np.genfromtxt(f'{folder}/mc.log')
    nsteps = data[:, 0]
    ener   = data[:, 4]
    cH     = data[:, 7]

    axs[0].plot(nsteps, ener, color=color, lw=0.8, label=label)
    axs[1].plot(nsteps, cH / N * 100, color=color, lw=0.8, label=label)

    half = len(cH) // 2
    print(f"{label:12s}  mean H conc. (last 50%): "
          f"{cH[half:].mean()/N*100:.1f} %")

axs[0].set_xlabel('MC steps')
axs[0].set_ylabel('Energy (eV)')
axs[0].set_title('Total energy')
axs[0].legend()

axs[1].set_xlabel('MC steps')
axs[1].set_ylabel('H / metal (%)')
axs[1].set_title('H concentration')
axs[1].legend()

plt.show()


---
## Part 3 — Parameter sensitivity

Explore how GCMC parameters affect the result.
For each experiment, modify the relevant parameter in the run script,
re-run, and plot the H concentration trace.

| Parameter | TurboGAP keyword | Suggested values to try |
|-----------|-----------------|--------------------------|
| Max displacement | `mc_move_max` | 0.05, 0.20, 0.80 Å |
| Temperature | `temperature` | 200, 300, 500 K |
| Move ratio (ins:disp) | `mc_moves` | insertion-only vs mixed |
| Chemical potential | `mu` | vary by ±0.2 eV |

> **Question:** Which parameter has the largest effect on the equilibrium
> H concentration? Which affects only the convergence speed?


In [ ]:
# Template for parameter sensitivity analysis.
# Modify 'folders' and 'labels' to match your run directories.

N = 79
folders = ['5.gcmc-1']   # add your new run directories here
labels  = ['baseline']   # add matching labels here

fig, axs = plt.subplots(1, 2, figsize=(8, 4), layout='constrained')

for folder, label in zip(folders, labels):
    data   = np.genfromtxt(f'{folder}/mc.log')
    nsteps = data[:, 0]
    ener   = data[:, 4]
    cH     = data[:, 7]

    axs[0].plot(nsteps, ener, lw=0.8, label=label)
    axs[1].plot(nsteps, cH / N * 100, lw=0.8, label=label)

    half = len(cH) // 2
    print(f"{label:20s}  mean H conc.: {cH[half:].mean()/N*100:.1f} %")

axs[0].set_xlabel('MC steps'); axs[0].set_ylabel('Energy (eV)')
axs[0].set_title('Energy convergence'); axs[0].legend()
axs[1].set_xlabel('MC steps'); axs[1].set_ylabel('H / metal (%)')
axs[1].set_title('H concentration'); axs[1].legend()
plt.show()
